# TVTSyn inference

From the repository root, install `pip install -e '.[pytorch,onnx,speaker]'` and `pip install ipykernel`. In VS Code, select the Python environment where you installed these packages.

Run the cells in order to download the model from Hugging Face and convert the sample audio with PyTorch and ONNX. Replace the sample paths with your own recordings. These examples use CPU.


In [1]:
import sys, importlib.util
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Notebook interpreter:", sys.executable)
if importlib.util.find_spec("tvtsyn_inference") is None:
    raise RuntimeError("Select the Python environment where you installed TVTSyn. Current interpreter: " + sys.executable)


Notebook interpreter: /data/mtseng/miniconda3/envs/TVTSyn/bin/python


In [2]:
from pathlib import Path
import numpy as np
from IPython.display import Audio, display
from tvtsyn_inference import TVTSyn

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
MODEL = 'warisqr007/TVTSyn'  # Or str(ROOT / 'artifacts/TVTSyn')
REVISION = '2d3147b8a465e1d9aad33a2a0e53956bf8ed32aa'
source = ROOT / 'samples/source.wav'
target = ROOT / 'samples/target.wav'

## PyTorch whole utterance and target embedding

In [3]:
pt = TVTSyn.from_pretrained(MODEL, backend='pytorch', revision=REVISION)
speaker = pt.encode_speaker(target)
output = ROOT / 'outputs'
output.mkdir(exist_ok=True)
np.save(output / 'target.npy', speaker)
display(Audio(pt.convert_file(source, speaker), rate=16000))

/data/mtseng/miniconda3/envs/TVTSyn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1070.11it/s]
/data/mtseng/miniconda3/envs/TVTSyn/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/data/mtseng/miniconda3/envs/TVTSyn/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:64: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Plea

## PyTorch streaming

In [4]:
display(Audio(pt.convert_file(source, speaker, mode='streaming'), rate=16000))

## ONNX streaming

With a saved target embedding, this backend does not require PyTorch or SpeechBrain.

In [5]:
onnx = TVTSyn.from_pretrained(MODEL, backend='onnx', revision=REVISION)
display(Audio(onnx.convert_file(source, output / 'target.npy'), rate=16000))

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 1120.07it/s]


## Incremental input and final flush

In [6]:
import soundfile as sf
x, sr = sf.read(source, dtype='float32')
stream = onnx.stream(output / 'target.npy', sample_rate=sr)
try:
    chunks = [stream.push(x[i:i+731]) for i in range(0, len(x), 731)]
    chunks.append(stream.flush())
finally:
    stream.close()
display(Audio(np.concatenate(chunks), rate=16000))